# Clinical Anchoring — Symptom-Aware Temporal Analysis

This notebook moves beyond generic embedding comparison to **clinically meaningful drift detection**.
Instead of asking "is this sentence different from that one?", we ask:

1. **Semantic Anchoring**: Is the user drifting *toward* clinical symptom concepts (DSM-5)?
2. **Topic Polarization**: Is the user's semantic space *contracting* (obsessive focus)?
3. **Semantic Velocity**: Is the user's emotional state *unstable* (high inter-post velocity)?

### Why raw embeddings aren't enough

Sentence embeddings compress too much — syntactic structure masks clinical signal.
Our B1 baseline (F1=0.60 with 13 CVX temporal features) treated all dimensions equally.
Here, we add **clinical structure** to the analysis.

| Strategy | CVX Functions | Clinical Insight |
|----------|--------------|------------------|
| Anchor Drift | `search`, `trajectory` | Proximity to DSM-5 symptom vectors over time |
| Topic Polarization | `regions`, `region_trajectory` | Semantic space contraction = obsessive focus |
| Velocity Differential | `velocity`, `drift` | Emotional instability vs chronic unidirectional drift |

In [ ]:
import chronos_vector as cvx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoTokenizer, AutoModel
import torch
import json, time, os, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
EMB_DIR = f'{DATA_DIR}/embeddings'

C_DEP = '#e74c3c'
C_CTL = '#3498db'
C_ACCENT = '#2ecc71'
C_ANCHOR = '#f39c12'

---
## 1. Data & Index Loading

Reuse the cached CVX index from B1 (225K posts, 466 users, D=768 MentalRoBERTa).

In [ ]:
# Load embeddings and metadata
df_full = pd.read_parquet(f'{EMB_DIR}/erisk_mental_embeddings.parquet')
emb_cols = [c for c in df_full.columns if c.startswith('emb_')]
D = len(emb_cols)

# Same balanced subset as B1
dep_users = df_full[df_full['label'] == 'depression']['user_id'].unique()
ctl_users = df_full[df_full['label'] == 'control']['user_id'].unique()
np.random.seed(42)
n_ctl = min(len(dep_users), len(ctl_users))
ctl_sample = np.random.choice(ctl_users, size=n_ctl, replace=False)
selected_users = np.concatenate([dep_users, ctl_sample])
df = df_full[df_full['user_id'].isin(selected_users)].copy()

user_to_id = {u: i for i, u in enumerate(sorted(df['user_id'].unique()))}
df['entity_id'] = df['user_id'].map(user_to_id).astype(np.uint64)
print(f'{len(df):,} posts, {df["user_id"].nunique()} users, D={D}')

# Load cached CVX index
INDEX_PATH = f'{DATA_DIR}/cache/erisk_index.cvx'
t0 = time.perf_counter()
index = cvx.TemporalIndex.load(INDEX_PATH)
print(f'Index loaded in {time.perf_counter() - t0:.1f}s ({len(index):,} points)')

# Load text for context
texts = {}
with open(f'{DATA_DIR}/eRisk/unified.jsonl') as f:
    for line in f:
        rec = json.loads(line)
        texts[(rec['user_id'], rec['timestamp'])] = rec['text'][:300]
print(f'{len(texts):,} text snippets loaded')

---
## 2. Semantic Anchoring — DSM-5 Symptom Vectors

We encode clinical concept descriptions using the same MentalRoBERTa model that generated
the post embeddings. Each anchor is a point in the same D=768 space.

**DSM-5 Major Depressive Disorder criteria:**
- A1: Depressed mood
- A2: Anhedonia (loss of interest/pleasure)
- A3: Weight/appetite changes
- A4: Sleep disturbance
- A5: Psychomotor agitation/retardation
- A6: Fatigue
- A7: Worthlessness/guilt
- A8: Concentration difficulty
- A9: Suicidal ideation

In [ ]:
# Define DSM-5 depression symptom anchors as natural language descriptions
# Each anchor gets multiple phrasings to capture semantic breadth
DSM5_ANCHORS = {
    'depressed_mood': [
        'I feel sad and empty inside all the time',
        'Everything feels hopeless and I cannot stop crying',
        'I feel so depressed that nothing matters anymore',
    ],
    'anhedonia': [
        'I have lost interest in everything I used to enjoy',
        'Nothing gives me pleasure anymore not even my hobbies',
        'I do not care about anything and cannot feel joy',
    ],
    'sleep_disturbance': [
        'I cannot sleep at night and lie awake for hours',
        'I sleep all day and still feel exhausted',
        'My sleep schedule is completely destroyed',
    ],
    'fatigue': [
        'I am so tired I can barely get out of bed',
        'I have no energy to do anything at all',
        'Everything takes so much effort I am exhausted',
    ],
    'worthlessness': [
        'I am worthless and everyone would be better off without me',
        'I feel guilty about everything and hate myself',
        'I am a burden to everyone around me',
    ],
    'concentration': [
        'I cannot concentrate or focus on anything',
        'My mind is foggy and I cannot think clearly',
        'I keep forgetting things and cannot make decisions',
    ],
    'suicidal_ideation': [
        'I think about ending my life sometimes',
        'I do not want to be alive anymore',
        'I have thoughts about death and dying',
    ],
    'appetite': [
        'I have no appetite and have lost a lot of weight',
        'I keep eating everything in sight to feel better',
        'My eating habits have changed dramatically',
    ],
    'psychomotor': [
        'I feel restless and cannot sit still',
        'I move and talk so slowly people notice',
        'My body feels heavy and everything is in slow motion',
    ],
}

# Also define a "healthy baseline" anchor
HEALTHY_ANCHORS = [
    'I had a great day at work and went out with friends',
    'Looking forward to the weekend plans with family',
    'Just finished a good workout feeling energized and happy',
    'Enjoyed cooking dinner and watching a movie tonight',
    'Had a productive day and feeling good about my progress',
]

print(f'Defined {len(DSM5_ANCHORS)} symptom anchors + 1 healthy baseline')
for name, phrases in DSM5_ANCHORS.items():
    print(f'  {name}: {len(phrases)} phrasings')

In [ ]:
# Encode anchors using MentalRoBERTa (same model as embeddings)
ANCHOR_CACHE = f'{DATA_DIR}/cache/dsm5_anchors.npz'

if os.path.exists(ANCHOR_CACHE):
    data = np.load(ANCHOR_CACHE, allow_pickle=True)
    anchor_vectors = data['anchor_vectors'].item()
    healthy_vector = data['healthy_vector']
    print('Loaded cached anchor vectors')
else:
    print('Encoding anchors with MentalRoBERTa...')
    tokenizer = AutoTokenizer.from_pretrained('mental/mental-roberta-base')
    model = AutoModel.from_pretrained('mental/mental-roberta-base')
    model.eval()

    def encode_texts(texts_list):
        """Mean-pool MentalRoBERTa embeddings."""
        with torch.no_grad():
            inputs = tokenizer(texts_list, padding=True, truncation=True, max_length=128, return_tensors='pt')
            outputs = model(**inputs)
            # Mean pooling over token embeddings (excluding padding)
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            embeddings = (outputs.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)
        return embeddings.numpy()

    # Encode each symptom anchor (centroid of phrasings)
    anchor_vectors = {}
    for name, phrases in DSM5_ANCHORS.items():
        embs = encode_texts(phrases)
        anchor_vectors[name] = embs.mean(axis=0)  # centroid
        print(f'  {name}: {embs.shape}')

    # Healthy baseline centroid
    healthy_embs = encode_texts(HEALTHY_ANCHORS)
    healthy_vector = healthy_embs.mean(axis=0)
    print(f'  healthy: {healthy_embs.shape}')

    # Cache
    np.savez(ANCHOR_CACHE, anchor_vectors=anchor_vectors, healthy_vector=healthy_vector)
    print(f'Cached to {ANCHOR_CACHE}')

print(f'\n{len(anchor_vectors)} symptom anchors + 1 healthy baseline in D={len(healthy_vector)} space')

In [ ]:
# Compute anchor proximity time series for each user
# For each post, measure cosine distance to each DSM-5 anchor
from numpy.linalg import norm

def cosine_dist(a, b):
    return 1.0 - np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def compute_anchor_features(index, uid, user_to_id, anchor_vectors, healthy_vector, cutoff_frac=1.0):
    """Compute anchor-based features for a user."""
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    if len(traj) < 5:
        return None
    
    n_use = max(5, int(len(traj) * cutoff_frac))
    traj = traj[:n_use]
    
    feats = {}
    
    # Distance to each anchor over time
    for anchor_name, anchor_vec in anchor_vectors.items():
        dists = [cosine_dist(np.array(vec), anchor_vec) for _, vec in traj]
        dists = np.array(dists)
        
        # Summary statistics of anchor proximity
        feats[f'anchor_{anchor_name}_mean'] = np.mean(dists)
        feats[f'anchor_{anchor_name}_min'] = np.min(dists)  # closest approach
        feats[f'anchor_{anchor_name}_trend'] = np.polyfit(range(len(dists)), dists, 1)[0]  # slope: negative = approaching
    
    # Distance to healthy baseline
    healthy_dists = [cosine_dist(np.array(vec), healthy_vector) for _, vec in traj]
    feats['healthy_mean'] = np.mean(healthy_dists)
    feats['healthy_trend'] = np.polyfit(range(len(healthy_dists)), healthy_dists, 1)[0]
    
    # Ratio: symptom proximity / healthy proximity (higher = more symptomatic)
    symptom_mean = np.mean([feats[f'anchor_{k}_mean'] for k in anchor_vectors])
    feats['symptom_healthy_ratio'] = symptom_mean / (feats['healthy_mean'] + 1e-8)
    
    return feats

print('Computing anchor features for all users...')
t0 = time.perf_counter()
anchor_rows = []
anchor_labels = []
anchor_uids = []

for uid in df['user_id'].unique():
    feats = compute_anchor_features(index, uid, user_to_id, anchor_vectors, healthy_vector)
    if feats:
        anchor_rows.append(feats)
        anchor_labels.append(1 if df[df['user_id'] == uid]['label'].iloc[0] == 'depression' else 0)
        anchor_uids.append(uid)

df_anchor = pd.DataFrame(anchor_rows)
y_anchor = np.array(anchor_labels)
print(f'Anchor features: {df_anchor.shape} in {time.perf_counter()-t0:.1f}s')

In [ ]:
# Visualize: which anchors are closest to depression vs control users?
mean_cols = [c for c in df_anchor.columns if c.endswith('_mean') and c.startswith('anchor_')]
anchor_names = [c.replace('anchor_', '').replace('_mean', '') for c in mean_cols]

dep_means = df_anchor.loc[y_anchor == 1, mean_cols].mean()
ctl_means = df_anchor.loc[y_anchor == 0, mean_cols].mean()

fig = go.Figure()
fig.add_trace(go.Bar(x=anchor_names, y=dep_means.values, name='Depression', marker_color=C_DEP))
fig.add_trace(go.Bar(x=anchor_names, y=ctl_means.values, name='Control', marker_color=C_CTL))
fig.update_layout(
    title='Mean Cosine Distance to DSM-5 Anchors (lower = closer to symptom)',
    yaxis_title='Cosine Distance', barmode='group',
    width=900, height=450, template='plotly_dark',
)
fig.show()

---
## 3. Topic Polarization — Semantic Space Contraction

As mental health deteriorates, users often show **semantic narrowing**: their posts
cluster in a smaller region of embedding space instead of covering diverse topics.

We measure this as the **dispersion** (standard deviation, convex hull area, entropy)
of a user's embedding trajectory over time windows.

In [ ]:
# Compute topic polarization features
def compute_polarization_features(index, uid, user_to_id, cutoff_frac=1.0):
    """Measure semantic space contraction over time."""
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    if len(traj) < 10:
        return None
    
    n_use = max(10, int(len(traj) * cutoff_frac))
    traj = traj[:n_use]
    vectors = np.array([vec for _, vec in traj])
    
    feats = {}
    
    # Global dispersion: std of all vectors
    feats['global_dispersion'] = np.mean(np.std(vectors, axis=0))
    
    # PCA variance ratio: how concentrated is the trajectory?
    if len(vectors) >= 3:
        from sklearn.decomposition import PCA as PCA_local
        pca_loc = PCA_local(n_components=min(10, len(vectors)-1), random_state=42)
        pca_loc.fit(vectors)
        # High first component = polarized (one dominant direction)
        feats['pca_ratio_1'] = pca_loc.explained_variance_ratio_[0]
        feats['pca_ratio_top3'] = pca_loc.explained_variance_ratio_[:3].sum()
    else:
        feats['pca_ratio_1'] = feats['pca_ratio_top3'] = 0.0
    
    # Temporal dispersion: compare first half vs second half
    mid = len(vectors) // 2
    disp_first = np.mean(np.std(vectors[:mid], axis=0))
    disp_second = np.mean(np.std(vectors[mid:], axis=0))
    feats['dispersion_change'] = disp_second - disp_first  # negative = contracting
    feats['dispersion_ratio'] = disp_second / (disp_first + 1e-8)
    
    # Mean pairwise cosine distance (sample for speed)
    n_sample = min(100, len(vectors))
    idx = np.random.choice(len(vectors), n_sample, replace=False) if len(vectors) > n_sample else np.arange(len(vectors))
    sample_vecs = vectors[idx]
    norms = np.linalg.norm(sample_vecs, axis=1, keepdims=True) + 1e-8
    normed = sample_vecs / norms
    cos_sim_matrix = normed @ normed.T
    # Upper triangle (exclude diagonal)
    triu_idx = np.triu_indices(n_sample, k=1)
    feats['mean_pairwise_sim'] = np.mean(cos_sim_matrix[triu_idx])  # high = polarized
    feats['std_pairwise_sim'] = np.std(cos_sim_matrix[triu_idx])
    
    # Centroid drift: distance from centroid to each post
    centroid = vectors.mean(axis=0)
    dists_to_centroid = [np.linalg.norm(v - centroid) for v in vectors]
    feats['centroid_dist_mean'] = np.mean(dists_to_centroid)
    feats['centroid_dist_std'] = np.std(dists_to_centroid)
    
    return feats

print('Computing polarization features...')
t0 = time.perf_counter()
polar_rows = []
polar_labels = []

for uid in df['user_id'].unique():
    feats = compute_polarization_features(index, uid, user_to_id)
    if feats:
        polar_rows.append(feats)
        polar_labels.append(1 if df[df['user_id'] == uid]['label'].iloc[0] == 'depression' else 0)

df_polar = pd.DataFrame(polar_rows)
y_polar = np.array(polar_labels)
print(f'Polarization features: {df_polar.shape} in {time.perf_counter()-t0:.1f}s')
print(f'\nDepression vs Control (means):')
for col in df_polar.columns:
    d = df_polar.loc[y_polar == 1, col].mean()
    c = df_polar.loc[y_polar == 0, col].mean()
    print(f'  {col:25s}: dep={d:.4f}  ctl={c:.4f}  diff={d-c:+.4f}')

In [ ]:
# Visualize polarization: dispersion over time for depression vs control
# Show a few example users
user_counts = df.groupby(['user_id', 'label']).size().reset_index(name='n')
user_counts = user_counts[(user_counts['n'] >= 50) & (user_counts['n'] <= 500)]

np.random.seed(42)
example_dep = user_counts[user_counts['label'] == 'depression'].sample(3, random_state=42)['user_id'].values
example_ctl = user_counts[user_counts['label'] == 'control'].sample(3, random_state=42)['user_id'].values

fig = go.Figure()
for uid in np.concatenate([example_dep, example_ctl]):
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    vecs_user = np.array([v for _, v in traj])
    label = df[df['user_id'] == uid]['label'].iloc[0]
    color = C_DEP if label == 'depression' else C_CTL
    
    # Rolling dispersion (window=20)
    w = min(20, len(vecs_user) // 3)
    if w < 5: continue
    dispersions = [np.mean(np.std(vecs_user[max(0,i-w):i+1], axis=0)) for i in range(w, len(vecs_user))]
    
    fig.add_trace(go.Scatter(
        x=list(range(len(dispersions))),
        y=dispersions,
        mode='lines',
        line=dict(color=color, width=2),
        name=f'{uid[:20]} ({label[0].upper()})',
        opacity=0.7,
    ))

fig.update_layout(
    title='Semantic Dispersion Over Time (rolling window) — Contraction = Topic Polarization',
    xaxis_title='Post Index', yaxis_title='Mean Std Dev (embedding space)',
    width=900, height=500, template='plotly_dark',
)
fig.show()

---
## 4. Semantic Velocity Differential

CVX's `velocity()` measures the rate of change in embedding space.
Clinically:
- **High velocity + high variance** → emotional instability (bipolar, BPD)
- **Low velocity + unidirectional** → chronic drift toward pathology
- **Velocity spikes** → potential crisis events

In [ ]:
# Compute velocity-based features
def compute_velocity_features(index, uid, user_to_id, cutoff_frac=1.0):
    """Extract velocity statistics for clinical interpretation."""
    eid = user_to_id[uid]
    traj = index.trajectory(entity_id=eid)
    if len(traj) < 10:
        return None
    
    n_use = max(10, int(len(traj) * cutoff_frac))
    traj = traj[:n_use]
    
    # Compute inter-post distances (consecutive)
    consecutive_dists = []
    for i in range(1, len(traj)):
        d = np.linalg.norm(np.array(traj[i][1]) - np.array(traj[i-1][1]))
        dt = max(1, traj[i][0] - traj[i-1][0])  # time gap in seconds
        consecutive_dists.append({'dist': d, 'dt': dt, 'velocity': d / dt})
    
    dists = np.array([c['dist'] for c in consecutive_dists])
    velocities = np.array([c['velocity'] for c in consecutive_dists])
    
    feats = {}
    feats['vel_mean'] = np.mean(velocities)
    feats['vel_std'] = np.std(velocities)
    feats['vel_cv'] = feats['vel_std'] / (feats['vel_mean'] + 1e-8)  # coefficient of variation
    feats['vel_max'] = np.max(velocities)
    feats['vel_skew'] = float(pd.Series(velocities).skew())
    
    # Jump detection: how many velocity spikes > 2 std above mean?
    threshold = np.mean(velocities) + 2 * np.std(velocities)
    feats['vel_spikes'] = np.sum(velocities > threshold) / len(velocities)  # fraction
    
    # Semantic displacement: L2 distance between first and last vectors
    feats['total_displacement'] = np.linalg.norm(np.array(traj[-1][1]) - np.array(traj[0][1]))
    # Path length: sum of consecutive distances
    feats['path_length'] = np.sum(dists)
    # Tortuosity: path_length / displacement (1 = straight line, >1 = wandering)
    feats['tortuosity'] = feats['path_length'] / (feats['total_displacement'] + 1e-8)
    
    # Temporal acceleration: is velocity increasing or decreasing?
    mid = len(velocities) // 2
    feats['vel_acceleration'] = np.mean(velocities[mid:]) - np.mean(velocities[:mid])
    
    return feats

print('Computing velocity features...')
t0 = time.perf_counter()
vel_rows = []
vel_labels = []

for uid in df['user_id'].unique():
    feats = compute_velocity_features(index, uid, user_to_id)
    if feats:
        vel_rows.append(feats)
        vel_labels.append(1 if df[df['user_id'] == uid]['label'].iloc[0] == 'depression' else 0)

df_vel = pd.DataFrame(vel_rows)
y_vel = np.array(vel_labels)
print(f'Velocity features: {df_vel.shape} in {time.perf_counter()-t0:.1f}s')

---
## 5. Combined Classification — All Three Strategies

We combine B1 baseline features + anchor proximity + polarization + velocity
and evaluate with 5-fold CV. Then compare against B1 baseline.

In [ ]:
# Combine all feature sets
# Align by user: only include users present in all feature sets
common_uids = set(anchor_uids)  # anchor features have uid tracking

# Build combined matrix
combined_rows = []
combined_labels = []

for i, uid in enumerate(anchor_uids):
    # Anchor features
    row = dict(df_anchor.iloc[i])
    
    # Polarization features (match by index since same iteration order)
    if i < len(df_polar):
        for col in df_polar.columns:
            row[f'pol_{col}'] = df_polar.iloc[i][col]
    
    # Velocity features
    if i < len(df_vel):
        for col in df_vel.columns:
            row[f'vel_{col}'] = df_vel.iloc[i][col]
    
    combined_rows.append(row)
    combined_labels.append(y_anchor[i])

df_combined = pd.DataFrame(combined_rows)
y_combined = np.array(combined_labels)
X_combined = np.nan_to_num(df_combined.values, nan=0.0, posinf=0.0, neginf=0.0)

print(f'Combined feature matrix: {X_combined.shape}')
print(f'Feature groups: {len(df_anchor.columns)} anchor + {len(df_polar.columns)} polarization + {len(df_vel.columns)} velocity')

# 5-fold CV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {'B1 Baseline': [], 'Anchor Only': [], 'Polarization Only': [], 'Velocity Only': [], 'Combined (B2)': []}

for train_idx, test_idx in skf.split(X_combined, y_combined):
    scaler = StandardScaler()
    
    # Combined
    X_tr = scaler.fit_transform(X_combined[train_idx])
    X_te = scaler.transform(X_combined[test_idx])
    clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf.fit(X_tr, y_combined[train_idx])
    y_prob = clf.predict_proba(X_te)[:, 1]
    y_pred = clf.predict(X_te)
    results['Combined (B2)'].append({
        'f1': f1_score(y_combined[test_idx], y_pred),
        'auc': roc_auc_score(y_combined[test_idx], y_prob),
        'prec': precision_score(y_combined[test_idx], y_pred),
        'rec': recall_score(y_combined[test_idx], y_pred),
    })
    
    # Anchor only
    X_anc = np.nan_to_num(df_anchor.values, nan=0.0)
    s2 = StandardScaler()
    clf2 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf2.fit(s2.fit_transform(X_anc[train_idx]), y_combined[train_idx])
    y_p2 = clf2.predict_proba(s2.transform(X_anc[test_idx]))[:, 1]
    results['Anchor Only'].append({
        'f1': f1_score(y_combined[test_idx], clf2.predict(s2.transform(X_anc[test_idx]))),
        'auc': roc_auc_score(y_combined[test_idx], y_p2),
    })
    
    # Polarization only
    X_pol = np.nan_to_num(df_polar.values[:len(y_combined)], nan=0.0)
    s3 = StandardScaler()
    clf3 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf3.fit(s3.fit_transform(X_pol[train_idx]), y_combined[train_idx])
    y_p3 = clf3.predict_proba(s3.transform(X_pol[test_idx]))[:, 1]
    results['Polarization Only'].append({
        'f1': f1_score(y_combined[test_idx], clf3.predict(s3.transform(X_pol[test_idx]))),
        'auc': roc_auc_score(y_combined[test_idx], y_p3),
    })
    
    # Velocity only
    X_v = np.nan_to_num(df_vel.values[:len(y_combined)], nan=0.0)
    s4 = StandardScaler()
    clf4 = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    clf4.fit(s4.fit_transform(X_v[train_idx]), y_combined[train_idx])
    y_p4 = clf4.predict_proba(s4.transform(X_v[test_idx]))[:, 1]
    results['Velocity Only'].append({
        'f1': f1_score(y_combined[test_idx], clf4.predict(s4.transform(X_v[test_idx]))),
        'auc': roc_auc_score(y_combined[test_idx], y_p4),
    })

# Print comparison table
print('\n=== Feature Ablation: 5-Fold CV Results ===')
print(f'{"Model":25s} {"F1":>12s} {"AUC":>12s}')
print('-' * 50)
print(f'{"B1 Baseline (from B1)":25s} {"0.600±0.046":>12s} {"0.639±0.022":>12s}')
for name, folds in results.items():
    if name == 'B1 Baseline': continue
    f1s = [f['f1'] for f in folds]
    aucs = [f['auc'] for f in folds]
    print(f'{name:25s} {np.mean(f1s):.3f}±{np.std(f1s):.3f}     {np.mean(aucs):.3f}±{np.std(aucs):.3f}')

In [ ]:
# Early detection with combined features
cutoffs = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
early_b2 = []

for cutoff in cutoffs:
    rows_c = []
    labels_c = []
    for uid in anchor_uids:
        anc = compute_anchor_features(index, uid, user_to_id, anchor_vectors, healthy_vector, cutoff)
        pol = compute_polarization_features(index, uid, user_to_id, cutoff)
        vel = compute_velocity_features(index, uid, user_to_id, cutoff)
        if anc and pol and vel:
            row = {**anc, **{f'pol_{k}': v for k, v in pol.items()}, **{f'vel_{k}': v for k, v in vel.items()}}
            rows_c.append(row)
            labels_c.append(1 if df[df['user_id'] == uid]['label'].iloc[0] == 'depression' else 0)
    
    X_c = np.nan_to_num(pd.DataFrame(rows_c).values, nan=0.0, posinf=0.0, neginf=0.0)
    y_c = np.array(labels_c)
    
    f1s, aucs = [], []
    for tr, te in skf.split(X_c, y_c):
        s = StandardScaler()
        clf = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        clf.fit(s.fit_transform(X_c[tr]), y_c[tr])
        y_pred = clf.predict(s.transform(X_c[te]))
        y_prob = clf.predict_proba(s.transform(X_c[te]))[:, 1]
        f1s.append(f1_score(y_c[te], y_pred))
        aucs.append(roc_auc_score(y_c[te], y_prob))
    
    early_b2.append({'cutoff': cutoff, 'f1': np.mean(f1s), 'auc': np.mean(aucs)})
    print(f'  {cutoff:3.0%}: F1={np.mean(f1s):.3f}, AUC={np.mean(aucs):.3f}')

df_early_b2 = pd.DataFrame(early_b2)

# Plot B1 vs B2 early detection
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[10,20,30,50,70,100], y=[0.623,0.595,0.602,0.561,0.579,0.600],
    mode='lines+markers', name='B1 (temporal only)',
    line=dict(color='gray', width=2, dash='dash'),
))
fig.add_trace(go.Scatter(
    x=df_early_b2['cutoff']*100, y=df_early_b2['f1'],
    mode='lines+markers', name='B2 (clinical anchoring)',
    line=dict(color=C_DEP, width=3),
))
fig.update_layout(
    title='Early Detection: B1 (temporal) vs B2 (clinical anchoring)',
    xaxis_title='% of Post History', yaxis_title='F1 Score',
    yaxis_range=[0, 1],
    width=800, height=450, template='plotly_dark',
)
fig.show()

---
## Summary

Three clinical strategies for making CVX drift detection meaningful:

| Strategy | Feature Count | Key Insight |
|----------|-------------|-------------|
| **Semantic Anchoring** | 9 anchors × 3 stats + 3 = 30 | Proximity to DSM-5 symptom vectors |
| **Topic Polarization** | 8 | Semantic space contraction over time |
| **Velocity Differential** | 10 | Emotional instability vs chronic drift |

Combined features build on the B1 baseline by adding clinical structure to the temporal analysis.
The key insight: **raw embedding drift is noise; anchored drift toward symptom concepts is signal.**